# 11 — Serialization, Reload, Inference, Latency, and Throughput (TensorFlow / Keras)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Training and inference are different workloads. Inference must reproduce preprocessing, load a versioned artifact, accept a stable input contract, produce deterministic output semantics, and satisfy latency/throughput constraints.

**Input contract:** one grayscale 28×28 image or a batch, converted to `float32`, flattened to 784, normalized to `[0,1]`. **Output contract:** 10 logits/probabilities plus predicted class; business systems may add confidence thresholds or abstention.


## Code walkthrough — reload the artifact and verify deterministic predictions
This cell intentionally starts from disk. Persisted artifacts are the unit of deployment. If a reloaded model does not reproduce the expected prediction path, packaging or serialization is broken.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)
from pathlib import Path
import urllib.request
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / "mnist.npz"
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_official_mnist():
    if not MNIST_PATH.exists():
        print("Downloading official MNIST archive to", MNIST_PATH)
        urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
    with np.load(MNIST_PATH) as data:
        return (data["x_train"], data["y_train"], data["x_test"], data["y_test"])


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    selected = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        selected.extend(rng.choice(candidates, size=per_class, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return (x[selected], y[selected])


def prepare_splits():
    x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_official_mnist()
    x_dev, y_dev = balanced_subset(x_train_raw, y_train_raw, per_class=600)
    x_test, y_test = balanced_subset(
        x_test_raw, y_test_raw, per_class=100, seed=SEED + 1
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev, y_dev, test_size=1000, random_state=SEED, stratify=y_dev
    )

    def transform(x):
        return x.reshape(len(x), -1).astype("float32") / 255.0

    return (
        transform(x_train),
        y_train,
        transform(x_val),
        y_val,
        transform(x_test),
        y_test,
    )


@tf.keras.utils.register_keras_serializable()
class MNISTMLP(tf.keras.Model):

    def __init__(self, hidden=64, **kwargs):
        super().__init__(**kwargs)
        self.hidden = hidden
        self.fc1 = tf.keras.layers.Dense(
            hidden, activation=None, kernel_initializer="he_normal"
        )
        self.act = tf.keras.layers.ReLU()
        self.fc2 = tf.keras.layers.Dense(
            10, activation=None, kernel_initializer="glorot_uniform"
        )

    def call(self, x, training=False):
        z1 = self.fc1(x)
        a1 = self.act(z1)
        logits = self.fc2(a1)
        return logits


model = MNISTMLP(hidden=64)
_ = model(tf.zeros((1, 784), dtype=tf.float32))
X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits()
model = tf.keras.models.load_model(ARTIFACT_DIR / "mnist_tensorflow_mlp.keras")
probs = tf.nn.softmax(model(X_test, training=False), axis=1).numpy()
print("first five true", y_test[:5])
print("first five predicted", probs[:5].argmax(1))
print("first five confidence", np.round(probs[:5].max(1), 3))

2026-09-07 18:55:18.753220: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


first five true [0 3 0 8 2]
first five predicted [0 3 0 8 2]
first five confidence [0.956 0.664 0.957 0.754 0.999]


## Code walkthrough — measure single-item latency and batched throughput
Microbenchmarks are noisy, but they teach an important systems principle: batching often improves throughput by amortizing framework and matrix-operation overhead. Production benchmarking should include warm-up, percentile latency, hardware specification and realistic concurrency.


In [2]:
import time

x1 = tf.convert_to_tensor(X_test[:1])
xb = tf.convert_to_tensor(X_test[:256])
for _ in range(20):
    model(xb, training=False)
start = time.perf_counter()
for _ in range(200):
    model(x1, training=False)
single_ms = (time.perf_counter() - start) / 200 * 1000
start = time.perf_counter()
for _ in range(100):
    model(xb, training=False)
batch_sec = time.perf_counter() - start
throughput = 100 * 256 / batch_sec
print(f"actual TensorFlow model single-item latency ~{single_ms:.3f} ms")
print(f"actual TensorFlow batch throughput ~{throughput:,.0f} images/sec")

actual TensorFlow model single-item latency ~1.157 ms
actual TensorFlow batch throughput ~160,274 images/sec


## Production implication
Model latency is only one component of request latency. Image decoding, feature lookup, network hops, serialization, model runtime and downstream policy all contribute. Monitor end-to-end service percentiles, not only notebook timing.
